In [ ]:

import pandas as pd

# Load dataset
df = pd.read_csv("/content/TabDPT_Results.csv")

results = []

# Traverse columns ending with _mean
for col in df.columns:
    if col.endswith("_mean"):

        # Corresponding SD column
        sd_col = col.replace("_mean", "_sd")

        if sd_col in df.columns:
            # Mean of dataset-level means
            mean_value = df[col].mean(skipna=True)

            # SD across dataset-level means
            sd_value = df[col].std(skipna=True, ddof=1)

            # Format to 4 decimal places
            formatted = f"{mean_value:.4f} ± {sd_value:.4f}"

            results.append({
                "Metric": col.replace("_mean", ""),
                "Result": formatted
            })

# Display results as a single column
output = pd.DataFrame(results)

print(output.to_string(index=False))

# Export to Excel
output.to_excel("tabdpt_aggregated_results.xlsx", index=False)

#print("Saved: aggregated_results.xlsx")

       Metric           Result
           DI  0.6608 ± 0.2490
          EOD  0.0819 ± 0.0589
          DPD  0.1635 ± 0.1276
         EqOD  0.1127 ± 0.0825
          PPD  0.1257 ± 0.1500
        AUROC  0.9197 ± 0.0665
  Sensitivity  0.8294 ± 0.1620
  Specificity  0.8707 ± 0.0915
          PPV  0.8383 ± 0.1313
          NPV  0.8802 ± 0.0802
          ECE  0.0753 ± 0.0466
          NLL  0.2825 ± 0.1700
        Brier  0.0895 ± 0.0568
    Cal_Slope  1.8599 ± 1.5880
Cal_Intercept -0.2472 ± 0.6019
    AUROC_Deg  0.0361 ± 0.0711
 Accuracy_Deg  0.0521 ± 0.1008
       F1_Deg  0.0427 ± 0.0770
    Flip_Rate  0.0750 ± 0.1161
         MAPC  0.0734 ± 0.0970


In [ ]:

import pandas as pd
import numpy as np
import re

# ==========================================
# 1. LOAD MASTER TABLE
# ==========================================

input_file = "/content/Table1_AggregatedResults.csv"
output_file = "model_ranking.xlsx"

df = pd.read_csv(input_file)

# ==========================================
# 2. DEFINE METRIC OPTIMIZATION DIRECTIONS
# ==========================================

# Higher is better
higher_is_better = [
    "AUROC",
    "Sensitivity",
    "Specificity",
    "PPV",
    "NPV"
]

# Lower is better (absolute differences / errors)
lower_is_better = [
    "EOD",
    "DPD",
    "EqOD",
    "PPD",
    "ECE",
    "NLL",
    "Brier",
    "AUROC_Deg",
    "Accuracy_Deg",
    "F1_Deg",
    "Flip_Rate",
    "MAPC"
]

# Closest to target is better
target_metrics = {
    "DI": 1.0,
    "Cal_Slope": 1.0,
    "Cal_Intercept": 0.0
}

# ==========================================
# 3. EXTRACT MEAN VALUES
# ==========================================

def extract_mean(value):
    if pd.isna(value):
        return np.nan

    # Handles values such as:
    # 0.9300 ± 0.0180
    # 0.9300 +/- 0.0180
    match = re.search(r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?", str(value))

    return float(match.group()) if match else np.nan


models = [col for col in df.columns if col != "Metrics"]

# Convert all mean ± SD entries into numeric means
numeric_data = df.copy()

for model in models:
    numeric_data[model] = numeric_data[model].apply(extract_mean)

# ==========================================
# 4. IDENTIFY WINNER FOR EACH METRIC
# ==========================================

metric_winners = []
model_wins = {model: [] for model in models}

for _, row in numeric_data.iterrows():

    metric = row["Metrics"]

    values = row[models].dropna()

    if values.empty:
        continue

    # Determine optimization direction
    if metric in higher_is_better:
        scores = values
        best_score = scores.max()

    elif metric in lower_is_better:
        scores = values.abs()
        best_score = scores.min()

    elif metric in target_metrics:
        target = target_metrics[metric]
        scores = (values - target).abs()
        best_score = scores.min()

    else:
        raise ValueError(f"Optimization direction not defined for {metric}")

    # Allow for numerical ties
    winners = scores[
        np.isclose(scores, best_score, rtol=1e-9, atol=1e-12)
    ].index.tolist()

    # Record winners
    for winner in winners:
        model_wins[winner].append(metric)

    metric_winners.append({
        "Metric": metric,
        "Winner": ", ".join(winners),
        "Winning Value": ", ".join(
            f"{values[w]:.4f}" for w in winners
        )
    })

# ==========================================
# 5. RANK MODELS BY NUMBER OF WINS
# ==========================================

ranking = []

for model, won_metrics in model_wins.items():

    ranking.append({
        "Model": model,
        "Metrics Won": len(won_metrics),
        "Won Metrics": ", ".join(won_metrics)
    })

ranking_df = pd.DataFrame(ranking)

ranking_df = ranking_df.sort_values(
    by=["Metrics Won", "Model"],
    ascending=[False, True]
).reset_index(drop=True)

ranking_df.insert(
    0, "Rank",
    ranking_df["Metrics Won"].rank(
        method="min", ascending=False
    ).astype(int)
)

metric_winners_df = pd.DataFrame(metric_winners)

# ==========================================
# 6. EXPORT TO EXCEL
# ==========================================

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:

    ranking_df.to_excel(
        writer,
        sheet_name="Model_Ranking",
        index=False
    )

    metric_winners_df.to_excel(
        writer,
        sheet_name="Metric_Winners",
        index=False
    )

print("\nMODEL RANKING")
print(ranking_df.to_string(index=False))

print("\nMETRIC WINNERS")
print(metric_winners_df.to_string(index=False))

print(f"\nResults saved to: {output_file}")



MODEL RANKING
 Rank   Model  Metrics Won                                              Won Metrics
    1   TabFM            8 EOD, AUROC, Sensitivity, NPV, ECE, NLL, Brier, Cal_Slope
    2 LogiReg            6    PPD, AUROC_Deg, Accuracy_Deg, F1_Deg, Flip_Rate, MAPC
    3  TabDPT            2                                                  DI, DPD
    3  TabPFN            2                                      EqOD, Cal_Intercept
    5  TabICL            1                                                      PPV
    5 XGBoost            1                                              Specificity

METRIC WINNERS
       Metric  Winner Winning Value
           DI  TabDPT        0.6608
          EOD   TabFM        0.0787
          DPD  TabDPT        0.1635
         EqOD  TabPFN        0.1123
          PPD LogiReg        0.0980
        AUROC   TabFM        0.9327
  Sensitivity   TabFM        0.8582
  Specificity XGBoost        0.8829
          PPV  TabICL        0.8565
          NPV   TabFM

In [ ]:

import pandas as pd
import numpy as np

# ============================================================
# 1. FILE PATHS
# ============================================================

input_file = "/content/TabDPT_Results.csv"
output_file = "TabDPT_Composite_Scores.xlsx"

# ============================================================
# 2. LOAD CSV DATA
# ============================================================

df = pd.read_csv(input_file)

# ============================================================
# 3. REQUIRED METRICS
#    Only the selected _mean columns are used.
# ============================================================

required_columns = [
    # Fairness
    "DI_mean",
    "DPD_mean",

    # Predictive Performance
    "AUROC_mean",
    "Sensitivity_mean",

    # Probabilistic Calibration
    "Brier_mean",
    "ECE_mean",

    # Robustness
    "AUROC_Deg_mean",
    "F1_Deg_mean"
]

# Check required columns
missing_columns = [
    col for col in required_columns if col not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

# Convert metrics to numeric
for col in required_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Check missing or invalid values
if df[required_columns].isna().any().any():
    raise ValueError(
        "Missing or non-numeric values found in the required "
        "_mean columns. Please check your CSV file."
    )

# ============================================================
# 4. VALIDATE METRIC RANGES
# ============================================================

# Metrics expected in [0, 1]
bounded_columns = [
    "DI_mean",
    "DPD_mean",
    "AUROC_mean",
    "Sensitivity_mean",
    "Brier_mean",
    "ECE_mean"
]

for col in bounded_columns:
    if not df[col].between(0, 1).all():
        raise ValueError(
            f"{col} contains values outside [0, 1]."
        )

# Degradation metrics expected in [-1, 1]
degradation_columns = [
    "AUROC_Deg_mean",
    "F1_Deg_mean"
]

for col in degradation_columns:
    if not df[col].between(-1, 1).all():
        raise ValueError(
            f"{col} contains values outside [-1, 1]."
        )

# ============================================================
# 5. FAIRNESS COMPOSITE SCORE (FCS)
# ============================================================

df["FCS"] = (
    df["DI_mean"]
    + (1 - df["DPD_mean"])
) / 2

# ============================================================
# 6. PREDICTIVE PERFORMANCE COMPOSITE SCORE (PPCS)
# ============================================================

df["PPCS"] = (
    df["AUROC_mean"]
    + df["Sensitivity_mean"]
) / 2

# ============================================================
# 7. PROBABILISTIC CALIBRATION COMPOSITE SCORE (PCCS)
# ============================================================

df["PCCS"] = (
    (1 - df["Brier_mean"])
    + (1 - df["ECE_mean"])
) / 2

# ============================================================
# 8. ROBUSTNESS COMPOSITE SCORE (RCS)
# ============================================================

df["RCS"] = (
    (1 - df["AUROC_Deg_mean"].abs())
    + (1 - df["F1_Deg_mean"].abs())
) / 2

# ============================================================
# 9. FINAL CLINICAL MODEL EVALUATION SCORE (CMES)
#
# Weights:
# Fairness                 = 1.0
# Predictive Performance   = 1.0
# Calibration              = 1.5
# Robustness               = 1.0
#
# Total weight = 4.5
# ============================================================

df["CMES"] = (
    df["FCS"]
    + df["PPCS"]
    + (1.5 * df["PCCS"])
    + df["RCS"]
) / 4.5

# ============================================================
# 10. EXPORT FIVE COMPOSITE SCORES
# ============================================================

output_columns = [
    "FCS",
    "PPCS",
    "PCCS",
    "RCS",
    "CMES"
]

output_df = df[output_columns].copy()

output_df.to_excel(
    output_file,
    index=False,
    float_format="%.6f"
)

# ============================================================
# 11. DISPLAY RESULTS
# ============================================================

print("\nComposite score calculation completed!")
print(f"Output saved to: {output_file}")

print("\nComposite scores:")
print(output_df.round(6).to_string(index=False))

print(f"\nNumber of datasets processed: {len(df)}")



Composite score calculation completed!
Output saved to: TabDPT_Composite_Scores.xlsx

Composite scores:
     FCS     PPCS     PCCS      RCS     CMES
0.654136 0.937365 0.920786 0.994562 0.881609
0.834090 0.934530 0.929735 0.992876 0.923577
0.708111 0.740962 0.848814 0.996414 0.826380
0.808778 1.000000 0.999908 0.987084 0.954605
0.894292 0.924368 0.930596 0.980998 0.932345
0.926812 0.735224 0.866959 0.992148 0.878805
0.418614 0.868694 0.895269 0.997267 0.806107
0.505107 0.693524 0.864743 0.993273 0.775337
0.907463 0.914704 0.934358 0.793432 0.892697
0.829259 0.996449 0.984764 0.856610 0.924325

Number of datasets processed: 10


In [ ]:

# ============================================================
# 1. INSTALL PACKAGES
# ============================================================

!pip install -q pandas numpy scipy statsmodels openpyxl

# ============================================================
# 2. IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np

from scipy.stats import friedmanchisquare, wilcoxon, rankdata
from itertools import combinations
from statsmodels.stats.multitest import multipletests

# ============================================================
# 3. DEFINE INPUT FILES
# ============================================================

files = {
    "LogiReg": "/content/LogiReg_Composite_Scores.xlsx",
    "TabDPT": "/content/TabDPT_Composite_Scores.xlsx",
    "TabFM": "/content/TabFM_Composite_Scores.xlsx",
    "TabICL": "/content/TabICL_Composite_Scores.xlsx",
    "TabPFN": "/content/TabPFN_Composite_Scores.xlsx",
    "XGBoost": "/content/XGBoost_Composite_Scores.xlsx"
}

models = list(files.keys())

# ============================================================
# 4. LOAD AND VALIDATE CMES VALUES
# ============================================================

model_data = {}

for model, path in files.items():

    df = pd.read_excel(path)

    if "CMES" not in df.columns:
        raise ValueError(
            f"CMES column not found in {model} file."
        )

    if len(df) != 10:
        raise ValueError(
            f"{model}: Expected 10 datasets, found {len(df)}."
        )

    scores = pd.to_numeric(df["CMES"], errors="coerce")

    if scores.isna().any() or not np.isfinite(scores).all():
        raise ValueError(
            f"{model}: Missing or invalid CMES values."
        )

    if not scores.between(0, 1).all():
        raise ValueError(
            f"{model}: CMES values must be between 0 and 1."
        )

    model_data[model] = scores.reset_index(drop=True)

print("All six model files loaded successfully.")

# IMPORTANT:
# The rows must correspond to the same datasets
# in exactly the same order across all six files.

# ============================================================
# 5. CREATE DATA MATRIX
# ============================================================

scores_df = pd.DataFrame(model_data)

print("\nCMES scores across the 10 datasets:")
print(scores_df.round(6).to_string())

# ============================================================
# 6. FRIEDMAN TEST
# ============================================================

statistic, p_value = friedmanchisquare(
    *[scores_df[model].values for model in models]
)

n = len(scores_df)
k = len(models)

# Kendall's W effect size
kendalls_w = statistic / (n * (k - 1))

friedman_result = pd.DataFrame([{
    "Test": "Friedman test",
    "Outcome": "CMES",
    "Number_of_Datasets": n,
    "Number_of_Models": k,
    "Friedman_Statistic": statistic,
    "Degrees_of_Freedom": k - 1,
    "p_value": p_value,
    "Kendalls_W": kendalls_w,
    "Significant": p_value < 0.05
}])

print("\n========== FRIEDMAN TEST ==========")
print(friedman_result.round(6).to_string(index=False))

# ============================================================
# 7. CALCULATE MEAN RANKS AND DESCRIPTIVE STATISTICS
# ============================================================

# Higher CMES = better rank
ranks = scores_df.apply(
    lambda row: rankdata(-row.values, method="average"),
    axis=1
)

ranks_df = pd.DataFrame(
    ranks.tolist(),
    columns=models
)

rank_results = pd.DataFrame({
    "Model": models,
    "Mean_CMES": [scores_df[m].mean() for m in models],
    "SD_CMES": [scores_df[m].std(ddof=1) for m in models],
    "Mean_Rank": [ranks_df[m].mean() for m in models]
}).sort_values("Mean_Rank")

print("\n========== MODEL RANKING ==========")
print(rank_results.round(6).to_string(index=False))

# ============================================================
# 8. POST-HOC WILCOXON TESTS
#    Only if Friedman test is significant
# ============================================================

posthoc_results = []

if p_value < 0.05:

    print("\nFriedman test is significant.")
    print("Performing pairwise Wilcoxon tests...")

    for model1, model2 in combinations(models, 2):

        x = scores_df[model1].values
        y = scores_df[model2].values

        differences = x - y

        if np.allclose(differences, 0):
            statistic_w = 0.0
            p_w = 1.0

        else:
            result = wilcoxon(
                x,
                y,
                alternative="two-sided",
                zero_method="wilcox",
                method="auto"
            )

            statistic_w = result.statistic
            p_w = result.pvalue

        posthoc_results.append({
            "Model_1": model1,
            "Model_2": model2,
            "Mean_Difference": np.mean(differences),
            "Median_Difference": np.median(differences),
            "Wilcoxon_Statistic": statistic_w,
            "p_value": p_w
        })

    posthoc_df = pd.DataFrame(posthoc_results)

    # Holm correction across 15 pairwise comparisons
    posthoc_df["Holm_Adjusted_p"] = multipletests(
        posthoc_df["p_value"],
        method="holm"
    )[1]

    posthoc_df["Significant"] = (
        posthoc_df["Holm_Adjusted_p"] < 0.05
    )

    print("\n========== POST-HOC RESULTS ==========")
    print(posthoc_df.round(6).to_string(index=False))

else:

    print(
        "\nFriedman test is NOT statistically significant "
        "(p >= 0.05)."
    )

    print(
        "Post-hoc comparisons were not performed."
    )

    posthoc_df = pd.DataFrame()

# ============================================================
# 9. EXPORT RESULTS TO EXCEL
# ============================================================

output_file = "/content/CMES_Statistical_Results.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:

    scores_df.to_excel(
        writer,
        sheet_name="CMES_Scores",
        index=False
    )

    friedman_result.to_excel(
        writer,
        sheet_name="Friedman_Test",
        index=False
    )

    rank_results.to_excel(
        writer,
        sheet_name="Model_Ranking",
        index=False
    )

    posthoc_df.to_excel(
        writer,
        sheet_name="Posthoc_Wilcoxon",
        index=False
    )

print("\n======================================")
print("CMES STATISTICAL ANALYSIS COMPLETED")
print("======================================")
print(f"Results saved to: {output_file}")


All six model files loaded successfully.

CMES scores across the 10 datasets:
    LogiReg    TabDPT     TabFM    TabICL    TabPFN   XGBoost
0  0.855577  0.881609  0.882715  0.879848  0.878886  0.857239
1  0.907177  0.923577  0.931915  0.917343  0.903129  0.910670
2  0.782064  0.826380  0.831237  0.829341  0.854881  0.819012
3  0.929501  0.954605  0.955759  0.956926  0.952369  0.943720
4  0.933335  0.932345  0.938509  0.936278  0.929642  0.907379
5  0.853636  0.878805  0.887056  0.878530  0.884525  0.858877
6  0.789757  0.806107  0.793510  0.802705  0.801811  0.803842
7  0.746377  0.775337  0.774397  0.765072  0.742733  0.754762
8  0.841386  0.892697  0.900121  0.894256  0.886707  0.873430
9  0.910380  0.924325  0.915935  0.914395  0.921429  0.921164

========== FRIEDMAN TEST ==========
         Test Outcome  Number_of_Datasets  Number_of_Models  Friedman_Statistic  Degrees_of_Freedom  p_value  Kendalls_W  Significant
Friedman test    CMES                  10                 6          

In [ ]:
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side

# Create the DataFrame
data = {
    'Model': ['TabFM', 'TabDPT', 'TabICL', 'TabPFN', 'XGBoost', 'LogiReg'],
    'Mean_CMES': [0.881115, 0.879579, 0.877469, 0.875611, 0.865010, 0.854919],
    'SD_CMES': [0.062012, 0.059276, 0.061058, 0.062728, 0.059076, 0.065861],
    'Mean_Rank': [2.0, 2.4, 2.9, 3.8, 4.4, 5.5]
}

df = pd.DataFrame(data)

# Define output filepath
input_file = '/content/Table4_Friedman_Mean_Rank.xlsx'

# Export to Excel
df.to_excel(input_file, index=False, sheet_name='Model Ranking')

# Apply Excel formatting
wb = load_workbook(input_file)
ws = wb['Model Ranking']

# Header formatting
header_fill = PatternFill('solid', fgColor='1F4E78')
header_font = Font(bold=True, color='FFFFFF', size=11)

for cell in ws[1]:
    cell.fill = header_fill
    cell.font = header_font
    cell.alignment = Alignment(horizontal='center')

# Data formatting
for row in ws.iter_rows(min_row=2):
    for cell in row:
        cell.alignment = Alignment(horizontal='center')

    # Numeric precision
    row[1].number_format = '0.000000'
    row[2].number_format = '0.000000'
    row[3].number_format = '0.0'

# Column widths
column_widths = {
    'A': 15,
    'B': 18,
    'C': 18,
    'D': 18
}

for col, width in column_widths.items():
    ws.column_dimensions[col].width = width

# Add borders
thin_border = Border(
    bottom=Side(style='thin', color='D9E2F3')
)

for row in ws.iter_rows():
    for cell in row:
        cell.border = thin_border

# Save formatted workbook
wb.save(input_file)

# Verify file creation
import os
assert os.path.exists(input_file)

print("Excel table created successfully!")
print("Filepath:", input_file)

Excel table created successfully!
Filepath: /content/Table4_Friedman_Mean_Rank.xlsx


In [ ]:

import pandas as pd
import os
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side

# Create the DataFrame
data = {
    'Model_1': [
        'LogiReg', 'LogiReg', 'LogiReg', 'LogiReg', 'LogiReg',
        'TabDPT', 'TabDPT', 'TabDPT', 'TabDPT',
        'TabFM', 'TabFM', 'TabFM',
        'TabICL', 'TabICL',
        'TabPFN'
    ],
    'Model_2': [
        'TabDPT', 'TabFM', 'TabICL', 'TabPFN', 'XGBoost',
        'TabFM', 'TabICL', 'TabPFN', 'XGBoost',
        'TabICL', 'TabPFN', 'XGBoost',
        'TabPFN', 'XGBoost',
        'XGBoost'
    ],
    'Mean_Difference': [
        -0.024660, -0.026196, -0.022550, -0.020692, -0.010091,
        -0.001537, 0.002109, 0.003967, 0.014569,
        0.003646, 0.005504, 0.016106,
        0.001858, 0.012460,
        0.010602
    ],
    'Median_Difference': [
        -0.025137, -0.026698, -0.021483, -0.017461, -0.009585,
        -0.003005, 0.001018, 0.002809, 0.016087,
        0.002549, 0.003610, 0.020440,
        0.002760, 0.011768,
        0.010963
    ],
    'Wilcoxon_Statistic': [
        1.0, 0.0, 0.0, 6.0, 8.0,
        20.0, 18.0, 15.0, 0.0,
        9.0, 17.0, 3.0,
        20.0, 4.0,
        10.0
    ],
    'p_value': [
        0.003906, 0.001953, 0.001953, 0.027344, 0.048828,
        0.492188, 0.375000, 0.232422, 0.001953,
        0.064453, 0.322266, 0.009766,
        0.492188, 0.013672,
        0.083984
    ],
    'Holm_Adjusted_p': [
        0.046875, 0.029297, 0.029297, 0.246094, 0.390625,
        1.000000, 1.000000, 1.000000, 0.029297,
        0.451172, 1.000000, 0.107422,
        1.000000, 0.136719,
        0.503906
    ],
    'Significant': [
        True, True, True, False, False,
        False, False, False, True,
        False, False, False,
        False, False,
        False
    ]
}

df = pd.DataFrame(data)

# Define output filepath
input_file = '/content/Table5_FriedManTest_PostHoc_CompositeMetrics.xlsx'

# Export DataFrame to Excel
df.to_excel(
    input_file,
    index=False,
    sheet_name='Post-Hoc Results'
)

# Load workbook for formatting
wb = load_workbook(input_file)
ws = wb['Post-Hoc Results']

# Header formatting
header_fill = PatternFill('solid', fgColor='1F4E78')
header_font = Font(bold=True, color='FFFFFF', size=10)

for cell in ws[1]:
    cell.fill = header_fill
    cell.font = header_font
    cell.alignment = Alignment(horizontal='center', vertical='center')

# Data formatting
for row in ws.iter_rows(min_row=2):
    for cell in row:
        cell.alignment = Alignment(horizontal='center', vertical='center')

    # Numeric precision
    for col in [3, 4, 6, 7]:
        row[col - 1].number_format = '0.000000'

    row[4].number_format = '0.0'

    # Highlight statistically significant results
    if row[7].value is True:
        for cell in row:
            cell.fill = PatternFill('solid', fgColor='E2F0D9')
            cell.font = Font(color='006100')

# Column widths
column_widths = {
    'A': 15,
    'B': 15,
    'C': 20,
    'D': 22,
    'E': 22,
    'F': 15,
    'G': 20,
    'H': 15
}

for col, width in column_widths.items():
    ws.column_dimensions[col].width = width

# Add borders
thin_border = Border(
    bottom=Side(style='thin', color='D9E2F3')
)

for row in ws.iter_rows():
    for cell in row:
        cell.border = thin_border

# Freeze header row and enable filters
ws.freeze_panes = 'A2'
ws.auto_filter.ref = ws.dimensions

# Adjust row height
ws.row_dimensions[1].height = 25

# Save workbook
wb.save(input_file)

# Verify file creation
assert os.path.exists(input_file), "Excel file creation failed!"

print("Excel file created successfully!")
print("Total pairwise comparisons:", len(df))
print("Significant comparisons:", df['Significant'].sum())
print("Output filepath:", input_file)


Excel file created successfully!
Total pairwise comparisons: 15
Significant comparisons: 4
Output filepath: /content/Table5_FriedManTest_PostHoc_CompositeMetrics.xlsx
